# Day 09 · Sessions 管理：對話狀態、Rewind 與 Schema 遷移

> 第二部・裝備升級　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 09 - Sessions 管理：對話狀態、Rewind 與 Schema 遷移.md`

## 今天要學會

1. 操作 `Session` 的四個核心屬性與生命週期
2. 用 `EventActions(state_delta=...)` 正確改 state
3. 在四種 `SessionService` 之間切換

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. Session 的四個核心屬性

| 屬性 | 是什麼 | 一句話 |
|---|---|---|
| `id` | 這段對話的識別碼 | 誰是誰 |
| `app_name` / `user_id` | 歸屬 | 誰的對話 |
| `events` | 完整事件歷史 | **發生過什麼** |
| `state` | 一個 dict | **現在怎樣** |

`events` 是流水帳，`state` 是結算後的餘額。兩者用途完全不同。

In [2]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

APP, USER = "day09", "student"

svc = InMemorySessionService()
agent = LlmAgent(name="assistant", model=get_model(),
                 instruction="你是助理，用繁體中文簡短回答。")
runner = Runner(agent=agent, app_name=APP, session_service=svc)

session = await svc.create_session(app_name=APP, user_id=USER)
print(f"id       : {session.id}")
print(f"app/user : {session.app_name} / {session.user_id}")
print(f"events   : {len(session.events)}")
print(f"state    : {dict(session.state)}")

id       : 4a3fa948-d8c8-40cf-9847-6766301ed8b2
app/user : day09 / student
events   : 0
state    : {}


## 2. 生命週期：四個步驟

```
  create_session()  →  run_async() 產生事件  →  append_event() 累積
                                                 ↓
                                         get_session() 取回
```

注意 `append_event` 通常是 ADK 幫你做的，但你也可以自己呼叫（第 4 節會用到）。

In [3]:
await ask(runner, "我叫 Sean，在做一個叫 Falcon 的專案。", session_id=session.id)
await ask(runner, "專案代號是什麼？", session_id=session.id)

s = await svc.get_session(app_name=APP, user_id=USER, session_id=session.id)
print(f"兩輪對話後：events = {len(s.events)} 筆\n")
for ev in s.events:
    text = ""
    if ev.content and ev.content.parts:
        text = "".join(p.text or "" for p in ev.content.parts if p.text)[:40]
    print(f"  [{ev.author:10s}] {text}")

兩輪對話後：events = 4 筆

  [user      ] 我叫 Sean，在做一個叫 Falcon 的專案。
  [assistant ] 你好，Sean！很高興認識你。祝你的 Falcon 專案進展順利！有什麼我可以協
  [user      ] 專案代號是什麼？
  [assistant ] 專案代號是 Falcon。


### 📌 「記憶」沒有魔法

第二個問題答得出來，不是因為 agent 記得，而是因為**整包 `events` 被重送給模型**。

這直接解釋了兩件事：

1. **長對話越來越貴**——每一輪都在重送全部歷史（解法在 Day 10 壓縮、Day 11 快取）
2. **清空 `events` 就等於失憶**

## 3. State 的作用域前綴

key 的前綴決定它活多久、給誰看：

| 前綴 | 作用域 | 用途 |
|---|---|---|
| （無） | 這個 session | 本次對話的暫存 |
| `user:` | 這個使用者的所有 session | 個人偏好 |
| `app:` | 整個應用 | 全域設定 |
| `temp:` | **只活在這一次呼叫** | 中間結果、敏感資料 |

In [4]:
from google.adk.tools import ToolContext


def save_preference(key: str, value: str, tool_context: ToolContext) -> dict:
    """記住使用者的偏好設定。

    Args:
        key: 偏好項目。
        value: 偏好內容。
    """
    tool_context.state[f"user:{key}"] = value     # 跨 session
    tool_context.state["last_key"] = key          # 只有這個 session
    tool_context.state["app:version"] = "1.0"     # 全應用
    tool_context.state["temp:scratch"] = "不會被存下來"
    return {"ok": True, key: value}


pref_agent = LlmAgent(
    name="pref_agent", model=get_model(),
    instruction="使用者說出偏好就呼叫 save_preference，然後用一句話確認。",
    tools=[save_preference],
)
pref_runner = Runner(agent=pref_agent, app_name=APP, session_service=svc)

sid_a = await new_session(pref_runner)
print(await ask(pref_runner, "我希望你以後都用正式的語氣", session_id=sid_a))
print("\n--- session A 的 state ---")
print_state(await peek_state(pref_runner, sid_a))

已為您調整為正式的語氣，日後將依此標準為您服務。

--- session A 的 state ---
  last_key: tone
  app:version: 1.0
  user:tone: 正式


In [5]:
sid_b = await new_session(pref_runner)
print("--- 全新的 session B ---")
print_state(await peek_state(pref_runner, sid_b))

--- 全新的 session B ---
  app:version: 1.0
  user:tone: 正式


觀察三件事：

1. `temp:scratch` **完全不見了**
2. `user:tone` 和 `app:version` 出現在**全新的 session B** 裡
3. `last_key` 只留在 session A

這是 ADK 內建的，你不用自己寫同步邏輯。

## 4. ⚠️ 不要直接改 `session.state`

這是最常見的錯誤，而且它**在開發環境看起來是好的**。

In [6]:
s = await svc.get_session(app_name=APP, user_id=USER, session_id=sid_a)
s.state["hacked"] = "我直接改的"

again = await svc.get_session(app_name=APP, user_id=USER, session_id=sid_a)
print("InMemorySessionService：直接改之後讀得到嗎？", "hacked" in again.state)

InMemorySessionService：直接改之後讀得到嗎？ False


「讀得到」——因為 `InMemorySessionService` 回傳的是**同一個物件參考**。

但換成有序列化的後端就完全不是這樣了。實測：

In [7]:
import tempfile
from pathlib import Path

from google.adk.sessions.sqlite_session_service import SqliteSessionService

db = Path(tempfile.gettempdir()) / "day09_demo.db"
db.unlink(missing_ok=True)

# ⚠️ SqliteSessionService 收的是 db_path（純路徑），
#    不是 DatabaseSessionService 那種 "sqlite:///..." 的 URL。
disk = SqliteSessionService(db_path=str(db))
disk_runner = Runner(agent=agent, app_name=APP, session_service=disk)

sid_d = await new_session(disk_runner)
await ask(disk_runner, "你好", session_id=sid_d)

d1 = await disk.get_session(app_name=APP, user_id=USER, session_id=sid_d)
d1.state["hacked"] = "我直接改的"

d2 = await disk.get_session(app_name=APP, user_id=USER, session_id=sid_d)
print("SqliteSessionService：直接改之後讀得到嗎？", "hacked" in d2.state)
print("\n→ 同一段程式碼，換一個後端行為就不一樣。這種 bug 通常上線才會爆。")

SqliteSessionService：直接改之後讀得到嗎？ False

→ 同一段程式碼，換一個後端行為就不一樣。這種 bug 通常上線才會爆。


### 正確的兩種改法

In [8]:
from google.adk.events import Event, EventActions

# 改法一：在工具裡用 tool_context.state（第 3 節已示範）

# 改法二：用 EventActions(state_delta=...) 送一個事件
await disk.append_event(
    session=await disk.get_session(app_name=APP, user_id=USER, session_id=sid_d),
    event=Event(
        author="system",
        actions=EventActions(state_delta={"reviewed_by": "sean", "user:plan": "pro"}),
    ),
)

print("用 state_delta 寫入之後：")
print_state(await peek_state(disk_runner, sid_d))

用 state_delta 寫入之後：
  reviewed_by: sean
  user:plan: pro


`state_delta` 的好處是**它本身也是一個事件**——
誰、在什麼時候、改了什麼，都留在 `events` 裡查得到。

這是 ADK 把「狀態變更」當成事件的核心設計，也是稽核與 Rewind 的基礎。

In [9]:
final = await disk.get_session(app_name=APP, user_id=USER, session_id=sid_d)
print("事件流裡的 state 變更紀錄：")
for ev in final.events:
    if ev.actions and ev.actions.state_delta:
        print(f"  [{ev.author}] {dict(ev.actions.state_delta)}")

事件流裡的 state 變更紀錄：
  [system] {'reviewed_by': 'sean', 'user:plan': 'pro'}


## 5. 四種 SessionService

全部是同一個介面，**換實作不用改 agent**。

In [10]:
import google.adk.sessions as sessions_mod

print("google.adk.sessions 的 __all__:")
print(" ", sessions_mod.__all__)

google.adk.sessions 的 __all__:
  ['BaseSessionService', 'DatabaseSessionService', 'InMemorySessionService', 'Session', 'State', 'StateSchemaError', 'VertexAiSessionService']


### ⚠️ 兩個很容易撞到的坑

In [11]:
# 坑 1：SqliteSessionService 不在 __all__ 裡
try:
    from google.adk.sessions import SqliteSessionService  # noqa: F401
    print("直接 import 成功")
except ImportError as exc:
    print(f"❌ 坑 1：{exc}")
    print("   → 要從子模組 import：")
    print("     from google.adk.sessions.sqlite_session_service import SqliteSessionService")

❌ 坑 1：cannot import name 'SqliteSessionService' from 'google.adk.sessions' (/Users/linshihuan/Dev/github/adk_tutor/.venv/lib/python3.13/site-packages/google/adk/sessions/__init__.py)
   → 要從子模組 import：
     from google.adk.sessions.sqlite_session_service import SqliteSessionService


In [12]:
# 坑 2：兩個很像的類別，參數名不一樣
import inspect

from google.adk.sessions import DatabaseSessionService
from google.adk.sessions.sqlite_session_service import SqliteSessionService as Sqlite

print("SqliteSessionService  :", inspect.signature(Sqlite.__init__))
print("DatabaseSessionService:", inspect.signature(DatabaseSessionService.__init__))
print()
print("❌ 坑 2：一個吃 db_path（純路徑），一個吃 db_url（sqlite:///...）。傳錯就 TypeError。")

SqliteSessionService  : (self, db_path: 'str')
DatabaseSessionService: (self, db_url: 'str | None' = None, db_engine: 'AsyncEngine | None' = None, **kwargs: 'Any') -> 'None'

❌ 坑 2：一個吃 db_path（純路徑），一個吃 db_url（sqlite:///...）。傳錯就 TypeError。


### ⚠️ 坑 3：`DatabaseSessionService` 需要**非同步**驅動

這個坑最難猜。`DatabaseSessionService` 底層走的是 SQLAlchemy 的 asyncio 擴充，
所以 URL 必須指定一個 **async driver**。用一般的同步 URL 會失敗：

In [13]:
for url in (f"sqlite:///{db}", f"sqlite+aiosqlite:///{db}"):
    try:
        DatabaseSessionService(db_url=url)
        print(f"  ✅ {url}")
    except Exception as exc:
        print(f"  ❌ {url}")
        print(f"     {str(exc).splitlines()[0][:90]}")

  ❌ sqlite:////tmp/day09_demo.db
     Failed to create database engine for URL 'sqlite:////tmp/day09_demo.db'
  ✅ sqlite+aiosqlite:////tmp/day09_demo.db


錯誤訊息的根因藏在裡面一層：

> *The asyncio extension requires an async driver to be used.
> The loaded 'pysqlite' is not async.*

各資料庫的 async driver 對照：

| 資料庫 | 同步 URL（❌） | async URL（✅） |
|---|---|---|
| SQLite | `sqlite:///x.db` | `sqlite+aiosqlite:///x.db` |
| PostgreSQL | `postgresql://…` | `postgresql+asyncpg://…` |
| MySQL | `mysql://…` | `mysql+aiomysql://…` |

**這也解釋了 `SqliteSessionService` 為什麼要獨立存在**——
它幫你把 driver 的事處理掉了，你只要給一個路徑。

In [14]:
# 用正確的 async URL 讀同一個檔案，確認兩種實作互通
dburl = DatabaseSessionService(db_url=f"sqlite+aiosqlite:///{db}")
via_url = await dburl.get_session(app_name=APP, user_id=USER, session_id=sid_d)
print("用 DatabaseSessionService 讀同一個檔案：")
print(f"  events = {len(via_url.events)} 筆")
print_state(dict(via_url.state))
print("\n→ 兩種實作共用同一份 schema，可以互通。")

用 DatabaseSessionService 讀同一個檔案：
  events = 3 筆
  reviewed_by: sean
  user:plan: pro

→ 兩種實作共用同一份 schema，可以互通。


### 選用建議

| 實作 | 存在哪 | 適合 |
|---|---|---|
| `InMemorySessionService` | 記憶體 | 開發、測試、單元測試 |
| `SqliteSessionService` | 本機 `.db` | 單機小專案、CLI 工具 |
| `DatabaseSessionService` | 任何 SQLAlchemy 後端 | 正式環境 |
| `VertexAiSessionService` | Google Cloud | 上雲、託管 |

> ⚠️ **Cloud Run 陷阱（Day 30 會再提）**：預設的 `InMemory*` 在每次冷啟動
> 都會清空。部署到無狀態環境一定要換成有持久化的後端。

## 6. 資料庫裡長什麼樣

In [15]:
import sqlite3

con = sqlite3.connect(db)
tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
print("SQLite 的表：")
for t in tables:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:14s} {n} 筆")
con.close()

SQLite 的表：
  app_states     0 筆
  user_states    1 筆
  sessions       1 筆
  events         3 筆
  adk_internal_metadata 1 筆


注意 `app_states` 和 `user_states` 是**獨立的表**——
這就是 `app:` / `user:` 前綴能跨 session 的實作方式。

## 7. Rewind 與 Schema 遷移

兩個版本限定的功能，知道存在就好：

| 功能 | 版本 | 做什麼 |
|---|---|---|
| **Rewind** | Python v1.17.0+ | 把 session 倒回某個事件之前（重試、除錯） |
| **Migrate** | Python v1.22.1+ | session schema 版本升級 |

Rewind 之所以可行，正是因為**所有狀態變更都是事件**（第 4 節）——
倒回去只要丟掉後面的事件、重算 state 就好。

In [16]:
import subprocess

adk = Path(sys.executable).parent / "adk"
r = subprocess.run([str(adk), "migrate", "--help"], capture_output=True, text=True, timeout=120)
print(r.stdout or r.stderr)

Usage: adk migrate [OPTIONS] COMMAND [ARGS]...

  ADK migration commands.

Options:
  --help  Show this message and exit.

Commands:
  session  Migrates a session database to the latest schema version.



In [17]:
db.unlink(missing_ok=True)
print("已清理")

已清理


## 8. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| 改了 `session.state` 但沒生效 | 直接改物件不會被記錄；用 `tool_context.state` 或 `state_delta` |
| 開發正常、上線 state 全丟 | 同上，只是 InMemory 剛好回傳同一個物件參考 |
| `ImportError: SqliteSessionService` | 它不在 `__all__`，要從子模組 import |
| `TypeError: unexpected keyword 'db_url'` | `SqliteSessionService` 吃 `db_path`，別跟 `DatabaseSessionService` 搞混 |
| `Failed to create database engine for URL 'sqlite:///…'` | `DatabaseSessionService` 要 **async driver**：`sqlite+aiosqlite://` |
| 部署到 Cloud Run 後對話全忘 | 用了 `InMemorySessionService`，冷啟動就清空 |
| 長對話成本暴增 | `events` 每輪重送 → Day 10 壓縮 / Day 11 快取 |

## 9. 動手練習

1. 把第 3 節的 `user:` 前綴拿掉，確認 session B 讀不到了。
2. 用 `state_delta` 連寫三次同一個 key，然後從 `events` 把變更歷史印出來——
   這就是稽核軌跡的雛形。
3. 把第 4 節的 `SqliteSessionService` 換成 `DatabaseSessionService`
   （注意參數名不同，而且要用 `sqlite+aiosqlite://`），確認行為一致。

## 本日回顧

- **Session = `events`（發生過什麼）+ `state`（現在怎樣）**，用途不同。
- **「記憶」就是 events 整包重送**——長對話變貴的根源。
- **State 前綴決定作用域**：`user:` / `app:` 跨 session，`temp:` **不會被存下來**。
- **⚠️ 不要直接改 `session.state`**。InMemory 下看起來有效，換後端就失效。
  正確做法是 `tool_context.state` 或 `EventActions(state_delta=...)`。
- **⚠️ 三個容易撞到的坑**：`SqliteSessionService` 不在 `__all__`（要從子模組 import）、
  它的參數是 `db_path` 不是 `db_url`、而 `DatabaseSessionService` 需要
  **async driver**（`sqlite+aiosqlite://`、`postgresql+asyncpg://`）。
- **換持久化只要換 SessionService**，agent 程式碼不動。

---
**下一天 → `../day10_context_compaction/`**